# Built-in MCP Server Toggles in `/configure mcp`

**Status:** Approved design; specification authored through the notebook MCP with executed gates
**Design epic:** `bd-ixbyj`
**Decision optimizations:** `sol_f7726bdfa2b54da4` (notebook-only ranking — USER-OVERRIDDEN to all-builtins), `sol_a660279275f74ed7` (guardrail: confirm dialog, 6/6), `sol_c013dfee25984d42` (per-tool question: staged per-module spur-mcp, 9/10)

Extends the shipped `/configure mcp` pane (gateway commits e46dfa87b..4ccede39f, probe commits 341c0f284, 42dc329b9) with visibility, probing, and runtime toggles for SPUR's built-in MCP servers: `spur-mcp`, `notebook`, `spur-worker-mcp`.


## Problem and grounded evidence

Built-ins are invisible to `/configure mcp` and immutable at runtime:

- `brain_mcp_servers` (`crates/spur-core/src/notebook.rs`) hardcodes `spur-mcp` (HTTP callback server) + the `notebook` stdio proxy; `RESERVED_MCP_SERVER_NAMES` blocks user entries with those names.
- The only existing opt-out is compile-time: the `disable-notebook-mcp` cargo feature (`crates/spur-core/src/notebook.rs:411`).
- Worker dispatch gates the curated `spur-worker-mcp` catalog through the per-delegation `enable_worker_mcp: Option<bool>` flag (default on; `crates/spur-core/src/orchestrator/worker_mcp.rs:136`).
- The probe module (`crates/spur-mcp/src/probe.rs`) can probe any `McpServerEntry`; built-ins need transient synthesized entries (notebook: resolved binary + nonce socket; spur-mcp: runtime url; worker: url + delegation token when live).

## Decisions

1. **All three built-ins toggleable** — user override of the solver's notebook-only ranking (`sol_f7726bdfa2b54da4`, which weighted boundary preservation at 5+5). Accepted consequences, on record: disabling `spur-mcp` removes delegation/plan/solve tools from brain sessions; disabling `spur-worker-mcp` opts workers out of the curated catalog.
2. **Guardrail: TUI confirmation dialog** when disabling `spur-mcp` (`sol_a660279275f74ed7`, 6/6: footgun_mitigation=3, config_freedom=2, impl_simplicity=1). No schema-level "min one enabled" constraint — hand-edited TOML stays free.
3. **Per-tool toggles: staged out** (`sol_c013dfee25984d42`): platform constraint — SPUR serves `spur-mcp` (ToolRegistry can filter advertised tools) but cannot filter `notebook`/remote servers' tools without a new MCP filtering-proxy component. Follow-up: per-**module** toggles under `builtin_overrides.spur_mcp_modules` (nests without schema break). Per-tool-for-all rejected.

## Injection semantics

| Built-in | Injected iff | Notes |
|---|---|---|
| `spur-mcp` | `spur_mcp_enabled` (default true) | toggle → confirm dialog → SAVE-APPLY |
| `notebook` | `notebook_enabled AND NOT compile_disabled` | runtime flag meets the existing `disable-notebook-mcp` feature — either off ⇒ skipped (gate below) |
| `spur-worker-mcp` | worker dispatch reads `worker_mcp_enabled` as the **default** for `enable_worker_mcp` | per-delegation flag still overrides |


In [ ]:
flowchart TD
    SPEC["`@spec MCP-NOTEBOOK-INJECTION
@type Status = enum[injected, skipped_runtime, skipped_compile]
@input notebook_enabled: Bool
@input compile_disabled: Bool
@output status: Status
@requires PRE: true`"]

    INJ["`@branch INJECTED
@when not compile_disabled and notebook_enabled
@ensures INJ_STATUS: status = injected`"]

    RT["`@branch SKIPPED_RUNTIME
@when not compile_disabled and (not notebook_enabled)
@ensures RT_STATUS: status = skipped_runtime`"]

    CP["`@branch SKIPPED_COMPILE
@when compile_disabled
@ensures CP_STATUS: status = skipped_compile`"]

    CHECK["`@verify NOTEBOOK_DETERMINISTIC: prove determinism
@verify NOTEBOOK_COVERAGE: prove partition_coverage
@verify NOTEBOOK_EXCLUSIVE: prove partition_exclusive
@verify NOTEBOOK_STATUSES: witness each status`"]

    SPEC --> INJ --> CHECK
    SPEC --> RT --> CHECK
    SPEC --> CP --> CHECK


## Components

### 1. Config (`crates/spur-acp/src/config/mod.rs`)

```rust
#[derive(Debug, Clone, Copy, PartialEq, Eq, Serialize, Deserialize)]
#[serde(rename_all = "snake_case")]
pub enum BuiltinMcpServer { SpurMcp, Notebook, SpurWorkerMcp }

#[derive(Debug, Clone, Default, PartialEq, Eq, Serialize, Deserialize)]
#[serde(default)]
pub struct BuiltinMcpOverridesConfig {
    pub spur_mcp_enabled: bool,     // default true via Default impl
    pub notebook_enabled: bool,     // default true
    pub worker_mcp_enabled: bool,   // default true
}
```

Nested on `McpServersConfig.builtin_overrides` (same `"mcp"` section; skip_serializing when all-true). New `ConfigPatch::BuiltinMcpToggle { server: BuiltinMcpServer, enabled: bool }`; `apply()` writes the matching field — the enum makes illegal keys unrepresentable (no free-form name map to validate).

### 2. Injection (`crates/spur-core`)

- `brain_mcp_servers(spur_mcp_url, socket_nonce, user, builtins)`: skip `spur-mcp` entry when `!spur_mcp_enabled`; notebook arm honors the meet semantics of the gate above (compile feature OR runtime flag); user entries unchanged.
- Worker dispatch: `build_worker_mcp_servers_with` callers pass `config.mcp_servers.builtin_overrides.worker_mcp_enabled` as the `enable_worker_mcp` default — per-delegation explicit flags still win.
- All three orchestrator call sites thread the overrides (they already pass `&self.config.mcp_servers`).

### 3. Pane (`crates/spur-tui/src/views/mcp_servers_tui.rs` + the browser)

- "Built-in servers" block above user entries: three rows (name, transport kind, runtime source, toggle state badge).
- `d` toggles → `spur-mcp` off routes through a confirmation dialog ("removes delegation/plan/solve tools from brain sessions — applies to next session") → SAVE-APPLY via existing `Action::ConfigSaveRequested { patch: ConfigPatch::BuiltinMcpToggle { .. } }`.
- `t` probes built-ins through transient synthesized `McpServerEntry`s (notebook: resolved binary path + nonce socket arg; spur-mcp: runtime url from app state; worker: url+token when a worker server is live, else "not running" hint).
- Disabled rows render dimmed with the next-session notice (SAVE-APPLY semantics unchanged — no live re-injection).

## Task decomposition

- **T1** `spur-acp`: `BuiltinMcpServer` + `BuiltinMcpOverridesConfig` + `ConfigPatch::BuiltinMcpToggle` + serde/tests (all-true skipped, round-trip, toggle apply per server).
- **T2** `spur-core`: injection gating (3 call sites + worker default) + tests incl. compile-feature meet semantics (existing 2-arg-era tests updated).
- **T3** `spur-tui`: built-in block, toggle + confirm dialog, transient probe entries, tests. T2 ∥ T3 after T1.

## Testing strategy

RED-GREEN per repo convention; solver gates re-run post-implementation (workflow family, applied-next-session traces) plus this notebook's cell re-run if guard sources change.

## Non-goals

- Per-tool / per-module toggles (staged follow-up; `spur_mcp_modules` nests later without schema break).
- Filtering proxy for notebook/remote servers.
- Live mid-session re-injection (toggles apply at next session start).

## Risks

- User disables everything, brain loses SPUR tools → mitigated by the confirm dialog (recorded decision; config stays free by design).
- Worker default change alters delegation behavior for configs that relied on default-on → the default stays true; only explicit toggles change behavior.
